# FoundationStereo 40 组数据批处理

此 notebook 复用 `fs_high_resolution.ipynb` 的 2448×2048 推理、CUDA 后处理和 Constrained Delaunay mesh 流程。它会扫描 `data/<group>/<capture>/` 下的全部数据组，并输出：

- `result/areas.json`：按 `data` 的两级目录结构保存每组 mesh 面积（单位 m²）；
- `result/<group>/<capture>/comparison.png`：从左到右为 rectified left、带绿色半透明 mask 的 rectified left、rectified right；
- `result/<group>/<capture>/mesh.ply`：带顶点颜色的 reconstruction triangle mesh。

先顺序运行全部单元。模型仅加载一次，随后连续处理 40 组数据。

## 1. 环境、输入发现与输出目录

In [1]:
%load_ext autoreload
%autoreload 2

from fs_high_resolution_utils import *

REPO_ROOT, FS_ROOT, DEVICE = bootstrap_notebook(globals())
DATA_ROOT = REPO_ROOT / 'data'
RESULT_ROOT = REPO_ROOT / 'result'
EXPECTED_WIDTH, EXPECTED_HEIGHT = 2448, 2048
REQUIRED_INPUTS = ('left.png', 'right.png', 'calibration.json', 'masks/left_mask.png')

def capture_sort_key(path):
    """Sort numeric capture folders as 0, 1, ..., 9."""
    try:
        return (0, int(path.name))
    except ValueError:
        return (1, path.name)

CAPTURES = sorted(
    (capture_dir
     for group_dir in sorted(DATA_ROOT.iterdir()) if group_dir.is_dir()
     for capture_dir in sorted(group_dir.iterdir(), key=capture_sort_key)
     if capture_dir.is_dir() and all((capture_dir / item).is_file() for item in REQUIRED_INPUTS)),
    key=lambda path: (path.parent.name, capture_sort_key(path)),
)

if len(CAPTURES) != 40:
    raise RuntimeError(f'期望在 {DATA_ROOT} 找到 40 组完整数据，实际找到 {len(CAPTURES)} 组。')

# Create all 4 × 10 output directories before inference starts.
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
for capture_dir in CAPTURES:
    (RESULT_ROOT / capture_dir.relative_to(DATA_ROOT)).mkdir(parents=True, exist_ok=True)

print(f'Repository: {REPO_ROOT}')
print(f'GPU: {torch.cuda.get_device_name(DEVICE)}')
print(f'Found {len(CAPTURES)} captures; outputs will be written to: {RESULT_ROOT}')
for capture_dir in CAPTURES:
    print(' -', capture_dir.relative_to(DATA_ROOT))


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Repository: /home/liu4000/Desktop/FS_realsense
GPU: NVIDIA RTX PRO 4000 Blackwell
Found 40 captures; outputs will be written to: /home/liu4000/Desktop/FS_realsense/result
 - Volunteer2_lower/0
 - Volunteer2_lower/1
 - Volunteer2_lower/2
 - Volunteer2_lower/3
 - Volunteer2_lower/4
 - Volunteer2_lower/5
 - Volunteer2_lower/6
 - Volunteer2_lower/7
 - Volunteer2_lower/8
 - Volunteer2_lower/9
 - Volunteer2_upper/0
 - Volunteer2_upper/1
 - Volunteer2_upper/2
 - Volunteer2_upper/3
 - Volunteer2_upper/4
 - Volunteer2_upper/5
 - Volunteer2_upper/6
 - Volunteer2_upper/7
 - Volunteer2_upper/8
 - Volunteer2_upper/9
 - Volunteer3_lower/0
 - Volunteer3_lower/1
 - Volunteer3_lower/2
 - Volunteer3_lower/3
 - Volunteer3_lower/4
 - Volunteer3_lower/5
 - Volunteer3_lower/6
 - Volunteer3_lower/7
 - Volunteer3_lower/8
 - Volunteer3_lower/9


## 2. 只加载一次：FoundationStereo 模型

In [2]:
VALID_ITERS, HIERA = 32, 1

model, CKPT_PATH = load_foundation_stereo_model(FS_ROOT, DEVICE)
print(f'Model loaded: {CKPT_PATH.name}')
print({key: f'{value:.1f} MiB' for key, value in gpu_memory_mib().items()})


Using cache found in /home/liu4000/.cache/torch/hub/facebookresearch_dinov2_main
/home/liu4000/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/liu4000/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/liu4000/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
using MLP layer as FFN


Model loaded: model_best_bp2.pth
{'allocated': '1431.9 MiB', 'reserved': '1436.0 MiB', 'peak_allocated': '1431.9 MiB', 'peak_reserved': '1436.0 MiB'}


## 3. 批量重建、导出 PNG / PLY 与面积 JSON

单组失败不会阻止其余数据继续处理；循环结束后会抛出包含失败目录的异常，并且只在全部 40 组成功时写出最终 `areas.json`。

In [3]:
REMOVE_INVISIBLE, DENOISE_CLOUD = True, True
Z_FAR_M = 1.0
DENOISE_MAX_NEIGHBOR_DISTANCE_M, DENOISE_MIN_NEIGHBORS = 0.01, 3
DENOISE_EDGE_MIN_NEIGHBORS = 2
DOWNSAMPLE_PIXELS = 4
MESH_MAX_EDGE_M = 0.02
MESH_MAX_DEPTH_JUMP_M = 0.01
PNG_NAME, PLY_NAME = 'comparison.png', 'mesh.ply'

def save_comparison_png(rectified_left, rectified_left_mask, rectified_right, output_path, capture_label):
    """Save the requested three-panel rectification and mask comparison."""
    green_overlay = np.zeros((*rectified_left_mask.shape, 4), dtype=np.float32)
    green_overlay[rectified_left_mask] = (0.0, 1.0, 0.0, 0.45)

    figure, axes = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    panels = (
        (rectified_left, 'Rectified left'),
        (rectified_left, 'Rectified left + green mask'),
        (rectified_right, 'Rectified right'),
    )
    for axis, (image, title) in zip(axes, panels):
        axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
    axes[1].imshow(green_overlay, interpolation='none')
    figure.suptitle(capture_label)
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(figure)

areas_m2 = {}
failures = []
total_inference_seconds = 0.0

for number, capture_dir in enumerate(CAPTURES, start=1):
    relative_path = capture_dir.relative_to(DATA_ROOT)
    group_name, capture_name = relative_path.parts
    output_dir = RESULT_ROOT / relative_path
    left_path = capture_dir / 'left.png'
    right_path = capture_dir / 'right.png'
    mask_path = capture_dir / 'masks' / 'left_mask.png'
    calibration_path = capture_dir / 'calibration.json'

    try:
        rectified_left, rectified_right, rectified_k, baseline_m, rectified_left_mask = (
            load_rectify_and_select_mask(
                globals(), left_path, right_path, mask_path, calibration_path,
                EXPECTED_WIDTH, EXPECTED_HEIGHT,
            )
        )
        inference = run_foundation_stereo_inference(
            model, rectified_left, rectified_right, DEVICE, VALID_ITERS, HIERA,
            warmup_runs=1 if number == 1 else 0,
        )
        total_inference_seconds += inference.seconds
        surface = postprocess_disparity_gpu(
            inference.disparity, rectified_k, baseline_m, rectified_left_mask, DEVICE, Z_FAR_M,
            remove_invisible=REMOVE_INVISIBLE, denoise=DENOISE_CLOUD,
            max_neighbor_distance_m=DENOISE_MAX_NEIGHBOR_DISTANCE_M,
            min_neighbors=DENOISE_MIN_NEIGHBORS,
            edge_min_neighbors=DENOISE_EDGE_MIN_NEIGHBORS,
        )
        mesh_result = build_constrained_mesh(
            surface.xyz_map, surface.keep_mask, rectified_left_mask, rectified_left,
            downsample_pixels=DOWNSAMPLE_PIXELS,
            max_edge_m=MESH_MAX_EDGE_M,
            max_depth_jump_m=MESH_MAX_DEPTH_JUMP_M,
        )
        save_triangle_mesh(mesh_result, output_dir / PLY_NAME)
        save_comparison_png(
            rectified_left, rectified_left_mask, rectified_right, output_dir / PNG_NAME,
            str(relative_path),
        )
        areas_m2.setdefault(group_name, {})[capture_name] = mesh_result.area_m2
        print(
            f'[{number:02d}/{len(CAPTURES)}] {relative_path}: '
            f'{mesh_result.area_m2:.6f} m² ({mesh_result.area_m2 * 1e4:.2f} cm²), '
            f'{inference.seconds:.2f} s'
        )
    except Exception as error:
        failures.append((str(relative_path), f'{type(error).__name__}: {error}'))
        print(f'[{number:02d}/{len(CAPTURES)}] FAILED {relative_path}: {error}')
    finally:
        for variable_name in (
            'rectified_left', 'rectified_right', 'rectified_k', 'baseline_m',
            'rectified_left_mask', 'inference', 'surface', 'mesh_result',
        ):
            globals().pop(variable_name, None)
        torch.cuda.empty_cache()
    
if failures:
    formatted_failures = '\n'.join(f' - {path}: {message}' for path, message in failures)
    raise RuntimeError(f'{len(failures)} group(s) failed; areas.json was not written:\n{formatted_failures}')

areas_path = RESULT_ROOT / 'areas.json'
areas_payload = {
    'unit': 'm²',
    'areas_m2': areas_m2,
}
areas_path.write_text(json.dumps(areas_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Completed {len(CAPTURES)} captures in {total_inference_seconds:.1f} s of timed inference.')
print(f'Areas JSON: {areas_path}')


/home/liu4000/Desktop/FS_realsense/FoundationStereo/core/foundation_stereo.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/home/liu4000/Desktop/FS_realsense/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/liu4000/Desktop/FS_realsense/FoundationStereo/core/geometry.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/liu4000/Desktop/FS_realsense/FoundationStereo/core/foundation_stereo.py:245: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precisio

[01/40] Volunteer2_lower/0: 0.019919 m² (199.19 cm²), 8.41 s
[02/40] Volunteer2_lower/1: 0.020342 m² (203.42 cm²), 8.31 s
[03/40] Volunteer2_lower/2: 0.021088 m² (210.88 cm²), 8.33 s
[04/40] Volunteer2_lower/3: 0.020587 m² (205.87 cm²), 8.36 s
[05/40] Volunteer2_lower/4: 0.020212 m² (202.12 cm²), 8.38 s
[06/40] Volunteer2_lower/5: 0.020666 m² (206.66 cm²), 8.39 s
[07/40] Volunteer2_lower/6: 0.020567 m² (205.67 cm²), 8.41 s
[08/40] Volunteer2_lower/7: 0.020331 m² (203.31 cm²), 8.42 s
[09/40] Volunteer2_lower/8: 0.020086 m² (200.86 cm²), 8.44 s
[10/40] Volunteer2_lower/9: 0.019560 m² (195.60 cm²), 8.46 s
[11/40] Volunteer2_upper/0: 0.089518 m² (895.18 cm²), 8.45 s
[12/40] Volunteer2_upper/1: 0.088593 m² (885.93 cm²), 8.45 s
[13/40] Volunteer2_upper/2: 0.090534 m² (905.34 cm²), 8.44 s
[14/40] Volunteer2_upper/3: 0.090867 m² (908.67 cm²), 8.45 s
[15/40] Volunteer2_upper/4: 0.091093 m² (910.93 cm²), 8.45 s
[16/40] Volunteer2_upper/5: 0.088869 m² (888.69 cm²), 8.45 s
[17/40] Volunteer2_upper